## 第一步：挂载 Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
# 确认 Drive 挂载成功
print('Drive 已挂载:', os.path.exists('/content/drive/MyDrive'))

Mounted at /content/drive
Drive 已挂载: True


## 第二步：克隆代码仓库

将下方 `GITHUB_REPO` 替换为你自己的仓库地址（`https://github.com/<你的用户名>/RecON.git`）。

In [2]:
GITHUB_REPO = 'https://github.com/Woomessi/Trackerless_3D_Ultrasound_Reconstruction.git'  # ← 修改此处
BRANCH = 'main'  # 如使用其他分支请修改
PROJECT_DIR = '/content/Trackerless_3D_Ultrasound_Reconstruction'

if not os.path.exists(PROJECT_DIR):
    !git clone --branch {BRANCH} {GITHUB_REPO} {PROJECT_DIR}
else:
    print('目录已存在，执行 git pull 更新...')
    !git -C {PROJECT_DIR} pull

%cd {PROJECT_DIR}
print('当前目录:', os.getcwd())

Cloning into '/content/Trackerless_3D_Ultrasound_Reconstruction'...
remote: Enumerating objects: 308, done.
remote: Counting objects: 100% (308/308), done.
remote: Compressing objects: 100% (188/188), done.
remote: Total 308 (delta 141), reused 275 (delta 110), pack-reused 0 (from 0)
Receiving objects: 100% (308/308), 26.05 MiB | 16.38 MiB/s, done.
Resolving deltas: 100% (141/141), done.
/content/Trackerless_3D_Ultrasound_Reconstruction
当前目录: /content/Trackerless_3D_Ultrasound_Reconstruction


## 第三步：安装依赖


In [3]:
# 确认 PyTorch 版本及 CUDA 可用性
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
##%%
# 安装额外依赖（timm、pyvista、h5py；opencv/scipy/numpy 已预装）
!pip install timm pyvista h5py --quiet

# 验证所有依赖可正常导入
import importlib
for pkg in ['cv2', 'numpy', 'scipy', 'timm', 'h5py', 'pyvista']:
    try:
        importlib.import_module(pkg)
        print(f'  ✓ {pkg}')
    except ImportError as e:
        print(f'  ✗ {pkg}: {e}')


PyTorch: 2.11.0+cu128
CUDA 可用: True
GPU: Tesla T4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.1/219.1 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.0/146.0 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.9/272.9 kB 22.5 MB/s eta 0:00:00
  ✓ cv2
  ✓ numpy
  ✓ scipy
  ✓ timm
  ✓ h5py
  ✓ pyvista


## 第四步：链接数据（直接读取 Drive，无需复制）

In [ ]:
import os, glob, h5py

DRIVE_DATA_DIR = '/content/drive/MyDrive/projects/3D_US_REC/datasets'  # ← 若 Drive 中路径不同请修改

# 检查 Drive 中数据是否存在
assert os.path.exists(DRIVE_DATA_DIR), f'未找到 Drive 数据目录: {DRIVE_DATA_DIR}'
assert os.path.exists(os.path.join(DRIVE_DATA_DIR, 'frames_transfs')), '缺少 frames_transfs 目录'
assert os.path.exists(os.path.join(DRIVE_DATA_DIR, 'calib_matrix.csv')), '缺少 calib_matrix.csv'

local_data_dir = os.path.join(PROJECT_DIR, 'data')
os.makedirs(local_data_dir, exist_ok=True)

DRIVE_FRAMES_DIR = os.path.join(DRIVE_DATA_DIR, 'frames_transfs')
LOCAL_FRAMES_DIR = os.path.join(local_data_dir, 'frames_transfs')

# 使用软链接直接指向 Drive，无需将数据集复制至本地
if os.path.lexists(LOCAL_FRAMES_DIR):
    print(f'软链接已存在，跳过: {LOCAL_FRAMES_DIR}')
else:
    os.symlink(DRIVE_FRAMES_DIR, LOCAL_FRAMES_DIR)
    print(f'已创建软链接: {LOCAL_FRAMES_DIR} → {DRIVE_FRAMES_DIR}')

# calib_matrix.csv 同样使用软链接
local_calib = os.path.join(local_data_dir, 'calib_matrix.csv')
if os.path.lexists(local_calib):
    os.remove(local_calib)
os.symlink(os.path.join(DRIVE_DATA_DIR, 'calib_matrix.csv'), local_calib)
print(f'链接: {local_calib} → Drive')

# 验证数据可读
h5_files = glob.glob(os.path.join(LOCAL_FRAMES_DIR, '**', '*.h5'), recursive=True)
print(f'\n找到 {len(h5_files)} 个 h5 文件')
with h5py.File(h5_files[0], 'r') as f:
    print(f'示例文件: {h5_files[0]}')
    print(f'  键: {list(f.keys())}')
    print(f'  frames shape: {f["frames"].shape}')
    print(f'  tforms shape: {f["tforms"].shape}')

## 第五步：配置模型保存路径


In [ ]:
def make_symlink(src, dst):
    if os.path.lexists(dst):
        os.remove(dst)
    os.symlink(src, dst)
    print(f'链接: {dst} → {src}')

DRIVE_SAVE_DIR = '/content/drive/MyDrive/projects/3D_US_REC/save'  # ← 可自定义
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

local_save_dir = os.path.join(PROJECT_DIR, 'save')
make_symlink(DRIVE_SAVE_DIR, local_save_dir)

print(f'检查点将保存至: {DRIVE_SAVE_DIR}')

## 第六步：Backbone 训练（帧间位移回归）

以下 cells 运行 `trial/train/main_train_RecON_backbone.py`，从头训练 RecON backbone。

**前置条件：**
1. 已完成第一步～第五步（Drive 挂载、代码克隆、依赖安装、save/ 软链接）
2. TUS 完整数据集（`train_part1/` 目录，含各 subject 子目录及 `.h5` 文件）已上传至 Drive

In [6]:
!git -C {PROJECT_DIR} pull

Already up to date.


In [ ]:
import json, os, glob

PROJECT_DIR = '/content/Trackerless_3D_Ultrasound_Reconstruction'

# 使用第四步创建的软链接（直接读取 Drive，无本地副本）
LOCAL_FRAMES_DIR = os.path.join(PROJECT_DIR, 'data', 'frames_transfs')

assert os.path.exists(LOCAL_FRAMES_DIR), (
    f'数据软链接未找到: {LOCAL_FRAMES_DIR}\n'
    '请先运行第四步完成软链接配置。'
)

# 统计 h5 文件数量
h5_files = glob.glob(os.path.join(LOCAL_FRAMES_DIR, '**', '*.h5'), recursive=True)
print(f'找到 {len(h5_files)} 个 .h5 文件')
assert h5_files, '目录下没有 .h5 文件，请检查路径和数据结构。'

# 更新 TUS_complete.json 中的数据路径
cfg_path = os.path.join(PROJECT_DIR, 'res/datasets/TUS_complete.json')
with open(cfg_path) as f:
    cfg = json.load(f)
cfg['paths']['h5'] = LOCAL_FRAMES_DIR
with open(cfg_path, 'w') as f:
    json.dump(cfg, f, indent=2)
print(f'已更新 TUS_complete.json -> h5: {LOCAL_FRAMES_DIR}')

# 检查 save 目录（检查点将写入此处）
save_dir = os.path.join(PROJECT_DIR, 'save')
print(f'检查点保存目录: {os.path.realpath(save_dir)}')

In [ ]:
##%%
%cd /content/Trackerless_3D_Ultrasound_Reconstruction

# 启动 backbone 训练（轮数和保存间隔由 res/run/hp_bk.json 控制）
!PYTHONPATH=/content/Trackerless_3D_Ultrasound_Reconstruction python trial/train/main_train_RecON_backbone.py